# Inventory Hybrid Algorithm
### Dataset: `dataset_after_preprocessing.xlsx` |

---
| Bagian | Isi |
|--------|-----|
| **[SETUP]** | Install & import library |
| **[CONFIG]** | Path file & parameter — **edit bagian ini** |
| **[PART 1]** | Fungsi load data |
| **[PART 2]** | Fungsi forecast (statistik + ML + hybrid) |
| **[PART 3]** | Fungsi klasifikasi ADI-CV² |
| **[PART 4]** | pipeline + evaluasi |
| **[RUN 1]** | Load & eksplorasi data |
| **[RUN 2]** | Forecast satu SKU |
| **[RUN 3]** | Running Model (Klasifikasi dan DDMRP Base) |
| **[RUN 4]** | Pipeline lengkap satu SKU|
| **[RUN 5]** | Sensitivity analysis |
| **[RUN 6]** | Export Excel |



## [SETUP] — Install & Import

In [39]:
!pip install openpyxl -q

In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, os, sys, math, json, pickle
from copy import deepcopy
from typing import Dict, List, Tuple, Optional
from scipy.optimize import minimize
from sklearn.linear_model import ElasticNet, BayesianRidge
from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,
                               HistGradientBoostingRegressor, ExtraTreesRegressor)
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.rcParams['figure.dpi'] = 110
sns.set_style('whitegrid')
print("✅ Library berhasil diimport")

✅ Library berhasil diimport


In [41]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## [CONFIG] — Konfigurasi Global ← Edit di sini

In [180]:
# ─── SESUAIKAN PATH INI ────────────────────────────────────────
FILE_PATH  = '/content/sample_data/Data 2.xlsx'
OUTPUT_DIR = '/content'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── PARAMETER ─────────────────────────────────────────────────
TRAIN_RATIO    = 0.80   # 80% train, 20% test
SL_TARGET      = 0.95   # target service level minimum
GA_POP_SIZE    = 30     # ukuran populasi GA
GA_N_GEN       = 80     # jumlah generasi GA

# ADI-CV² thresholds (Syntetos & Boylan 2005)
ADI_THRESHOLD  = 1.32
CV2_THRESHOLD  = 0.49
sku_number=100008503
# MOQ method: 'eoq' | 'dlt' | 'week' | 'none'
#MOQ_METHOD = 'eoq'

print("✅ Config loaded")
print(f"   File   : {FILE_PATH}")
print(f"   Output : {OUTPUT_DIR}")
#print(f"   MOQ    : {MOQ_METHOD}")

✅ Config loaded
   File   : /content/sample_data/Data 2.xlsx
   Output : /content


## [PART 1] — Data Loading

**Kolom sales:** `ID Item`, `Date`, `Demand `, `Sales Price Price After Discont`, `Promo Discount`
**Kolom master:** `Material Number`, `Lead Time_Days`, `Sales Price`, `Purchase Price`, `Holding Cost Rate/day`, `Lost Sale Rate/Each`, `Logistic Cost/Order`

**Catatan asumsi:**
- `IsPromo` dibuat dari `Promo Discount > 0`
- `PromoType` = `"NONE"` (tidak ada di data)  
- `Holding Cost Rate/day` sudah dalam format per hari (÷ 365 sudah dilakukan di sumber)
- `Pack Size` = 1 (tidak ada di data)
- `Std Dev LT` = 10% × DLT


In [181]:
def load_all_data(file_path=FILE_PATH):
    """Load sales dan master data dari Excel."""
    print(f"Loading: {file_path}")
    df_s = pd.read_excel(file_path, sheet_name='sales')
    df_m = pd.read_excel(file_path, sheet_name='sku_master')
    df_m['Initial Inventory'] = pd.to_numeric(df_m['Initial Inventory'], errors='coerce').fillna(0)

    # Bersihkan tipe data
    df_s['Date']     = pd.to_datetime(df_s['Date'])
    df_s['Demand ']  = pd.to_numeric(df_s['Demand '],  errors='coerce').fillna(0)
    df_s['Promo Discount'] = pd.to_numeric(df_s['Promo Discount'], errors='coerce').fillna(0)

    # Buat kolom turunan — TIDAK ADA di data asli
    df_s['IsPromo']         = df_s['Promo Discount'] > 0
    df_s['PromoType']       = 'NONE'          # tidak ada di dataset baru
    df_s['PromoDiscountPct']= df_s['Promo Discount']

    n_sku = df_s['ID Item'].nunique()
    print(f"✅ Sales    : {len(df_s):,} baris | {n_sku} SKU | "
          f"{df_s['Date'].min().date()} s/d {df_s['Date'].max().date()}")
    print(f"✅ Master   : {len(df_m)} SKU")
    return {'sales': df_s, 'master': df_m}


def get_sku_list(data, show=True):
    """Tampilkan daftar semua SKU."""
    df_s = data['sales']; df_m = data['master']
    grp = df_s.groupby('ID Item').agg(
        Grup        =('Nama Item','first'),
        Tgl_Mulai   =('Date','min'),
        Tgl_Akhir   =('Date','max'),
        Jml_Hari    =('Date','count'),
        Total_Demand=('Demand ','sum'),
        ADU         =('Demand ','mean'),
    ).reset_index()
    grp = grp.merge(df_m[['Material Number','Lead Time_Days','Logistic Cost/Order']],
                    left_on='ID Item', right_on='Material Number', how='left')
    grp = grp.drop(columns=['Material Number'])
    grp['ADU'] = grp['ADU'].round(1)
    grp['Tgl_Mulai'] = grp['Tgl_Mulai'].dt.date
    grp['Tgl_Akhir'] = grp['Tgl_Akhir'].dt.date
    if show:
        print("\n=== DAFTAR SKU ===")
        print(grp.to_string(index=False))
    return grp

def get_sku_demand(data, sku, start_date=None, end_date=None, verbose=True):
    df = data['sales']
    d  = df[df['ID Item'] == sku].copy()
    if d.empty:
        raise ValueError(f"SKU {sku} tidak ditemukan.")
    if start_date: d = d[d['Date'] >= pd.to_datetime(start_date)]
    if end_date:   d = d[d['Date'] <= pd.to_datetime(end_date)]

    d = d[['Date','Demand ','Sales Price Price After Discont',
            'IsPromo','PromoDiscountPct','PromoType']].copy()
    d.columns = ['Date','Demand','Price','IsPromo','PromoDiscountPct','PromoType']
    d = d.sort_values('Date').reset_index(drop=True)

    # Isi tanggal yang hilang
    dr = pd.date_range(d['Date'].min(), d['Date'].max(), freq='D')
    d  = pd.DataFrame({'Date':dr}).merge(d, on='Date', how='left')
    d['Demand']          = d['Demand'].fillna(0)
    d['IsPromo']         = d['IsPromo'].fillna(False)
    d['PromoDiscountPct']= d['PromoDiscountPct'].fillna(0.0)
    d['PromoType']       = d['PromoType'].fillna('NONE')
    d['Price']           = d['Price'].ffill()

    if verbose:
        print(f"✅ SKU {sku} | {d['Date'].min().date()} s/d {d['Date'].max().date()} | "
              f"{len(d)} hari | demand total: {d['Demand'].sum():,.3f} Pcs")
    return d.reset_index(drop=True)


def get_sku_params(data, sku):
    """Ambil parameter biaya & operasional satu SKU dari master (satuan CTN)."""
    df_m = data['master']
    row  = df_m[df_m['Material Number'] == sku]
    if row.empty:
        raise ValueError(f"SKU {sku} tidak ditemukan di master.")
    row = row.iloc[0]

    dlt            = int(row['Lead Time_Days'])
    initial_inventory = float(row.get('Initial Inventory', 0))

    # Skalakan harga & biaya ke satuan CTN
    price_ea  = float(row['Sales Price'])
    buy_ea    = float(row['Purchase Price'])
    hold_day_ea = float(row['Holding Cost Rate/day']) * price_ea
    penalty_ea = float(row['Lost Sale Rate/Each']) * price_ea

    # ==========================
    # Initial inventory dalam Pcs/EA
    # ==========================
    initial_inventory = float(row.get('Initial Inventory', 0))
    moq = int(row.get('MOQ', 1))
    qmax = parse_qmax(row.get('Qmax', None)) if 'Qmax' in row.index else None
    target_percentile = parse_target_percentile(
        row.get('Target Percentile', 0.95),
        default=0.95
    ) if 'Target Percentile' in row.index else 0.95

    #_hc  = hold_day_ea
    #_oc  = float(row['Logistic Cost/Order'])
    #_adu = float(row.get('ADU', 1)) if float(row.get('ADU', 0)) > 0 else 1
    #doc  = 0 #math.sqrt(2 * _oc * _adu / _hc) if _hc > 0 else 0

    return {
        'sku'                  : int(sku),
        'group'                : str(row['Material Group']),
        'dlt'                  : dlt,
        'lt_std'               : round(dlt * 0.10, 2),
        'pack_size'            : int(row.get('MOQ', 1)),
        'price_ea'             : round(price_ea, 2),
        'initial_inventory'    : round(initial_inventory, 4),
        'purchase_price'       : round(buy_ea, 2),
        'margin_pct'           : round((price_ea - buy_ea) / price_ea * 100, 1),
        'hold_rate_annual'     : round(float(row['Holding Cost Rate/day']) * 365, 4),
        'hold_cost_per_unit_day': round(hold_day_ea, 6),
        'lost_sale_rate'       : float(row['Lost Sale Rate/Each']),
        'penalty_per_unit'     : round(penalty_ea, 2),
        'order_cost'           : float(row['Logistic Cost/Order']),
        'moq'                  : int(row.get('MOQ', 1)),
        'qmax'                 : qmax,
        'target_percentile'    : target_percentile,
        #'doc'                  : round(doc, 2)
    }

print("✅ Part 1 — fungsi data loading siap")

✅ Part 1 — fungsi data loading siap


## [PART 3] — Klasifikasi ADI-CV²

In [182]:
VF_SPACE = {
    'SMOOTH': (0.10, 0.30),
    'ERRATIC': (0.25, 0.55),
    'INTERMITTENT': (0.30, 0.65),
    'LUMPY': (0.55, 1.00)
}

LTF_SPACE = {
    'SHORT': (0.61, 1.00),
    'MEDIUM': (0.41, 0.60),
    'LONG': (0.20, 0.40)
}

VF_GLOBAL_BOUNDS = (0.01, 3.00)
LTF_GLOBAL_BOUNDS = (0.01, 3.00)

def classify_sku(df_demand, params, verbose=True):
    """Hitung ADI, CV², kategori, method, dan search space VF/LTF."""

    s = df_demand['Demand'].values.astype(float)
    nz = s[s > 0]
    n = len(s)
    k = len(nz)

    adu = float(np.mean(s)) if n > 0 else 0.0

    if k == 0:
        adi = np.inf
        cv2 = 0.0
        cv2_all = 0.0
        cat = 'INTERMITTENT'
    else:
        adi = n / k
        mean_nz = np.mean(nz)

        cv2 = float((np.std(nz, ddof=1) / mean_nz) ** 2) if k > 1 and mean_nz > 0 else 0.0
        cv2_all = float((np.std(s, ddof=1) / adu) ** 2) if adu > 0 else 0.0

        if adi <= ADI_THRESHOLD and cv2 <= CV2_THRESHOLD:
            cat = 'SMOOTH'
        elif adi <= ADI_THRESHOLD and cv2 > CV2_THRESHOLD:
            cat = 'ERRATIC'
        elif adi > ADI_THRESHOLD and cv2 <= CV2_THRESHOLD:
            cat = 'INTERMITTENT'
        else:
            cat = 'LUMPY'

    # Method harus di luar blok if-else agar tetap terbentuk saat k == 0
    if cat in ['SMOOTH', 'ERRATIC']:
        method = 'DDMRP'
    else:
        method = 'DDMRP_CONDITIONAL'

    dlt = params['dlt']
    lt_cat = 'SHORT' if dlt <= 10 else ('MEDIUM' if dlt <= 25 else 'LONG')

    vf_lo, vf_hi = VF_SPACE.get(cat, (0.10, 1.00))
    ltf_lo, ltf_hi = LTF_SPACE.get(lt_cat, (0.41, 0.60))

    vf_init = (vf_lo + vf_hi) / 2
    ltf_init = (ltf_lo + ltf_hi) / 2


    result = {
        'sku': params['sku'],
        'group': params['group'],
        'dlt': dlt,
        'lt_category': lt_cat,

        'adu': round(adu, 2),
        'n_days': n,
        'n_nonzero': int(k),
        'n_zero': int((s == 0).sum()),

        'adi': round(adi, 4) if np.isfinite(adi) else np.inf,
        'cv2': round(cv2, 4),
        'cv2_all': round(cv2_all, 4),
        'category': cat,
        'method': method,

        # Qmax hanya ditampilkan, tidak dihitung di klasifikasi
        'qmax': params.get('qmax', None),
        'target_percentile': params.get('target_percentile', 0.95),

        'vf_low': vf_lo,
        'vf_high': vf_hi,
        'ltf_low': ltf_lo,
        'ltf_high': ltf_hi,

        'vf_init': round(vf_init, 4),
        'ltf_init': round(ltf_init, 4),
    }

    if verbose:
        emoji = {
            'SMOOTH': '🟢',
            'ERRATIC': '🟡',
            'INTERMITTENT': '🟠',
            'LUMPY': '🔴'
        }

        print(f"\n{'='*55}")
        print(f"  KLASIFIKASI SKU {params['sku']} | {params['group']}")
        print(f"{'='*55}")
        print(f"  ADI   = {adi:.4f}" if np.isfinite(adi) else "  ADI   = inf")
        print(f"  CV²   = {cv2:.4f}")
        print(f"  Kategori: {emoji.get(cat, '⚪')} {cat}")
        print(f"  Metode  : {method}")
        print(f"  DLT     : {dlt} hari ({lt_cat})")
        print(f"  qmax    : {params.get('qmax', None)}")
        print(f"  Target Percentile: {params.get('target_percentile', 0.95)}")
        print(f"  Search space: VF=[{vf_lo:.2f},{vf_hi:.2f}] | LTF=[{ltf_lo:.2f},{ltf_hi:.2f}]")
        #print(f"  Buffer awal : TOR={tor:.2f} | TOY={toy:.2f} | TOG={tog:.2f}")

    return result


print("✅ Part 3 — fungsi klasifikasi siap")

✅ Part 3 — fungsi klasifikasi siap


## [PART 4] — DDMRP Simulator

In [183]:
#-------------DDMRP Method----------------------#
###-------------------------------------------###


import numpy as np
import pandas as pd
import math

def apply_moq_qmax(q_raw, moq=0, qmax=None, enforce_moq=True):
    # Jika tidak ada kebutuhan order
    if q_raw is None or q_raw <= 0:
        return 0

    # Normalisasi MOQ
    moq = 0 if moq is None else moq

    if enforce_moq:
        # MOQ menjadi batas minimum order
        q = max(q_raw, moq)

        # Qmax hanya diterapkan jika Qmax >= MOQ
        # Jika Qmax < MOQ, tetap order sesuai MOQ
        if qmax is not None and qmax >= moq:
            q = min(q, qmax)

        return q

    else:
        q = q_raw
        if qmax is not None:
            q = min(q, qmax)
        return q
def simulate_ddmrp(
    demands,
    dates,
    vf,
    ltf,
    dlt,
    pack_size,
    unit_price,
    hold_cost_per_unit_day,
    order_cost,
    penalty_mult,
    forecast=None,
    verbose=False,
    qmax=None,
    initial_inventory=None,
    moq=0,
    qd_source="actual_demand",
):

    # =====================================================
    # 1. Persiapan data
    # =====================================================

    demands = np.asarray(demands, dtype=float)
    #forecast = np.asarray(forecast, dtype=float)
    dates = pd.Series(dates).reset_index(drop=True)

    n = len(demands)
    dlt = int(dlt)
    moq = max(int(moq), 0)

    # Forecast opsional
    if forecast is None:
        forecast = np.zeros(n)
    else:
        forecast = np.asarray(forecast, dtype=float)

    if len(forecast) < n:
        forecast = np.pad(
            forecast,
            (0, n - len(forecast)),
            mode="constant",
            constant_values=0
        )

    if qmax is not None:
        qmax = float(qmax)

    if initial_inventory is None or pd.isna(initial_inventory):
        raise ValueError("Initial Inventory wajib diisi.")

    # =====================================================
    # 2. Hitung parameter buffer DDMRP
    # =====================================================

    adu = float(np.mean(demands)) if n > 0 else 0.0
    ost = adu

    bzr = adu * dlt * ltf
    tor = bzr * vf
    yellow = adu * dlt
    toy = tor + yellow

    #doc_qty = 0 #doc if doc > 0 else adu * dlt
    green = max(bzr, moq)
    tog = toy + green

    # =====================================================
    # 3. Inisialisasi simulasi
    # =====================================================
    oh = float(initial_inventory)
    pipeline = {}
    rows = []

    # =========================
    # 4. Fungsi QD
    # =========================
    def compute_qd_at(t_index):
        """
        QD:
        - actual_demand : demand hari ini + demand aktual H+1..H+DLT yang > OST
        - forecast      : demand hari ini + forecast H+1..H+DLT yang > OST
        """
        if t_index >= n:
            return 0.0

        qd_val = float(demands[t_index])

        if qd_source == "actual_demand":
            for kk in range(1, dlt + 1):
                j = t_index + kk
                if j < len(demands) and demands[j] > ost:
                    qd_val += float(demands[j])

        elif qd_source == "forecast":
            for kk in range(1, dlt + 1):
                j = t_index + kk
                if j < len(forecast) and forecast[j] > ost:
                    qd_val += float(forecast[j])

        else:
            raise ValueError("qd_source harus 'actual_demand' atau 'forecast'.")

        return qd_val

    # =========================
    # 5. Simulasi harian
    # =========================
    for t in range(n):
        date = pd.Timestamp(dates.iloc[t]).normalize()

        # 1. Receipt masuk
        receipt = float(pipeline.pop(date, 0.0))
        oh += receipt

        # 2. On-order / pipeline
        op = sum(
            float(q) for d, q in pipeline.items()
            if 1 <= (pd.Timestamp(d).normalize() - date).days <= dlt
        )

        # 3. Inventory position
        ip = oh + op

        # 4. QD
        qd = compute_qd_at(t)

        # 5. NFE
        nfe = oh + op - qd


        # 6. Zona buffer

        if nfe <= tor:
            zone = "RED"
        elif nfe <= toy:
            zone = "YELLOW"
        else:
            zone = "GREEN"

        # 7. Keputusan order DDMRP standar
        q = 0
        q_raw = 0
        order_reason = "NO_ORDER"

        if nfe <= toy:
            q_raw = tog - nfe
            q = apply_moq_qmax(
                q_raw=q_raw,
                moq=moq,
                qmax=None,
                enforce_moq=True
            )

            if q > 0:
                order_reason = "DDMRP_STANDARD"

        # 8. Order masuk pipeline
        if q > 0:
            arr = (date + pd.Timedelta(days=dlt)).normalize()
            pipeline[arr] = pipeline.get(arr, 0.0) + float(q)

        # 9. Demand hari ini dipenuhi setelah keputusan order
        dem = float(demands[t])
        oh_before_demand = oh
        shipped = min(dem, oh)
        unmet = max(dem - shipped, 0.0)
        oh_end = oh - shipped

        # 10. Update OH untuk hari berikutnya
        oh = oh_end

        # 11. Total Biaya
        holding_cost = oh_end * hold_cost_per_unit_day
        order_cost_day = order_cost if q > 0 else 0
        penalty_cost = unmet * penalty_mult
        total_cost = holding_cost + order_cost_day + penalty_cost


        # 12. Simpan hasil harian
        rows.append({
            'date': date.date(),
            "method": "DDMRP",
            'demand': round(dem, 2),
            'forecast': round(float(forecast[t]) if t < len(forecast) else 0, 2),
            'receipt': round(receipt, 2),

            'OH_before_demand': round(oh_end + shipped, 2),
            'OH_end': round(oh_end, 2),
            'OP': round(op, 2),
            'IP': round(ip, 2),

            'QD': round(qd, 2),
            'NFE': round(nfe, 2),
            'TOR': round(tor, 2),
            'TOY': round(toy, 2),
            'TOG': round(tog, 2),
            'zone': zone,
            'q_raw': round(q_raw, 2),

            'order_reason': order_reason,
            'order_qty': int(q),

            'shipped': round(shipped, 2),
            'unmet': round(unmet, 2),

            'holding_cost': round(holding_cost, 2),
            'order_cost': round(order_cost_day, 2),
            'penalty_cost': round(penalty_cost, 2),
            'total_cost': round(total_cost, 2)
        })
    # =====================================================
    # 6. DataFrame hasil simulasi
    # =====================================================
    df = pd.DataFrame(rows)

    td = float(df['demand'].sum())
    ts = float(df['shipped'].sum())
    ns = int((df['unmet'] > 1e-6).sum())


    # =====================================================
    # 7. KPI
    # =====================================================
    kpi = {
        'method': "DDMRP",
        'vf': round(vf, 4),
        'ltf': round(ltf, 4),
        'adu': round(adu, 4),

        'bzr': round(bzr, 2),
        'tor': round(tor, 2),
        'toy': round(toy, 2),
        'tog': round(tog, 2),
        'initial_inventory': round(float(initial_inventory), 4),

        'fill_rate': round(ts / td, 4) if td > 0 else 1.0,
        'csl': round(1 - ns / n, 4) if n > 0 else 1.0,
        'n_stockout': ns,

        'total_cost': round(float(df['total_cost'].sum()), 0),
        'hold_cost': round(float(df['holding_cost'].sum()), 0),
        'order_cost': round(float(df['order_cost'].sum()), 0),
        'penalty_cost': round(float(df['penalty_cost'].sum()), 0),

        'n_orders': int((df['order_qty'] > 0).sum()),
        'total_order_qty': int(df['order_qty'].sum()),
        'avg_oh': round(float(df['OH_end'].mean()), 2),

        'df_detail': df
    }

    # =====================================================
    # 8. Print ringkasan
    # =====================================================
    if verbose:
        print(f"\n{'='*60}")
        print(f"SIMULASI DDMRP")
        print(f"{'='*60}")
        print(f"Initial Inventory = {initial_inventory}")
        print(f"ADU               = {adu:.4f}")
        print(f"VF                = {vf:.4f}")
        print(f"LTF               = {ltf:.4f}")
        print(f"TOR               = {tor:.2f}")
        print(f"TOY               = {toy:.2f}")
        print(f"TOG               = {tog:.2f}")
        print(f"Fill Rate         = {kpi['fill_rate']*100:.2f}%")
        print(f"CSL               = {kpi['csl']*100:.2f}%")
        print(f"Stockout Days     = {kpi['n_stockout']}")
        print(f"Jumlah Order      = {kpi['n_orders']}")
        print(f"Total Qty Order   = {kpi['total_order_qty']}")
        print(f"Total Cost        = Rp{kpi['total_cost']:,.0f}")

    return kpi

In [184]:
##------------------------DDMRP-CONDITIONAL METHOD----------------------------------##
##----------------------------------------------------------------------------------##
import numpy as np
import pandas as pd
import math

def compute_ltd_stats(demands, dlt, target_percentile=0.95):
    """
    Hitung rolling Lead Time Demand.
    target_percentile:
    0.90 = P90
    0.95 = P95
    0.99 = P99
    """

    s = np.asarray(demands, dtype=float)
    n = len(s)
    dlt = int(dlt)

    if n == 0:
        ltd = np.array([0.0])
    elif n < dlt:
        ltd = np.array([s.sum()])
    else:
        ltd = np.array([s[i:i + dlt].sum() for i in range(n - dlt + 1)])

    avg_ltd = float(np.mean(ltd)) if len(ltd) > 0 else 0.0

    p75_ltd = float(np.percentile(ltd, 75)) if len(ltd) > 0 else 0.0
    p90_ltd = float(np.percentile(ltd, 90)) if len(ltd) > 0 else 0.0
    p95_ltd = float(np.percentile(ltd, 95)) if len(ltd) > 0 else 0.0

    target_ltd = float(np.percentile(ltd, target_percentile * 100)) if len(ltd) > 0 else 0.0
    prob_ltd_pos = float(np.mean(ltd > 0)) if len(ltd) > 0 else 0.0

    ss_target = math.ceil(max(0, target_ltd - avg_ltd))
    ss_p90 = math.ceil(max(0, p90_ltd - avg_ltd))
    ss_p95 = math.ceil(max(0, p95_ltd - avg_ltd))

    return {
        "ltd_values": ltd,
        "avg_ltd": avg_ltd,
        "p75_ltd": p75_ltd,
        "p90_ltd": p90_ltd,
        "p95_ltd": p95_ltd,
        "target_ltd": target_ltd,
        "prob_ltd_pos": prob_ltd_pos,
        "ss_target": ss_target,
        "ss_p90": ss_p90,
        "ss_p95": ss_p95
    }

def apply_moq_qmax(q_raw, moq=0, qmax=None, enforce_moq=True):
    # Jika tidak ada kebutuhan order
    if q_raw is None or q_raw <= 0:
        return 0

    # Normalisasi MOQ
    moq = 0 if moq is None else moq

    if enforce_moq:
        # MOQ menjadi batas minimum order
        q = max(q_raw, moq)

        # Qmax hanya diterapkan jika Qmax >= MOQ
        # Jika Qmax < MOQ, tetap order sesuai MOQ
        if qmax is not None and qmax >= moq:
            q = min(q, qmax)

        return q

    else:
        q = q_raw
        if qmax is not None:
            q = min(q, qmax)
        return q

def simulate_ddmrp_conditional(
    demands,
    dates,
    vf,
    ltf,
    dlt,
    pack_size,
    unit_price,
    hold_cost_per_unit_day,
    order_cost,
    penalty_mult,
    verbose=False,
    forecast=None,
    target_level=None,
    qmax=None,
    initial_inventory=None,
    moq=0,
    target_percentile=0.95,
    use_qd_next_for_trigger=True,
    qd_source="actual_demand",
    target_stock_basis="ip",
    critical=False,          # tetap disediakan agar tidak error jika masih dipanggil
    min_stock_crit=1         # tidak dipakai dalam trigger final
):
    """
    Simulator DDMRP dan DDMRP Conditional.

    method:
    - "DDMRP"              : untuk SMOOTH dan ERRATIC
    - "DDMRP_CONDITIONAL"  : untuk INTERMITTENT dan LUMPY

    qd_source:
    - "forecast"      : QD memakai forecast masa depan
    - "actual_demand" : QD memakai demand aktual yang tersedia

    target_stock_basis:
    - "oh_before" : filter target memakai OH sebelum demand hari ini dikurangkan
    - "oh_after"  : filter target memakai OH setelah demand hari ini dikurangkan
    """
    # =====================================================
    # 1. Persiapan data
    # =====================================================
    demands = np.asarray(demands, dtype=float)
    dates = pd.Series(dates).reset_index(drop=True)

    n = len(demands)
    dlt = int(dlt)
    pack_size = max(int(pack_size), 1)
    moq = max(int(moq), 0)

    # Forecast opsional
    if forecast is None:
        forecast = np.zeros(n)
    else:
        forecast = np.asarray(forecast, dtype=float)

    if len(forecast) < n:
        forecast = np.pad(
            forecast,
            (0, n - len(forecast)),
            mode="constant",
            constant_values=0
        )

    if qmax is not None:
        qmax = float(qmax)

    if initial_inventory is None or pd.isna(initial_inventory):
        raise ValueError("Initial Inventory wajib diisi.")

    # =====================================================
    # 2. Hitung buffer DDMRP sebagai trigger awal
    # =====================================================
    adu = float(np.mean(demands)) if n > 0 else 0.0
    ost = adu
    bzr = adu * dlt * ltf
    tor = bzr * vf
    yellow = adu * dlt
    toy = tor + yellow

    green = max(bzr, moq)
    tog = toy + green

    # =================================================
    # 3. Hitung target level berbasis Lead Time Demand
    # =================================================

    ltd_stats = compute_ltd_stats(
        demands=demands,
        dlt=dlt,
        target_percentile=target_percentile
    )

    if target_level is None:
        target_level = math.ceil(ltd_stats['target_ltd'])
        ss_value = ltd_stats['ss_target']
    else:
        target_level = float(target_level)
        ss_value = None


    # =====================================================
    # 4. Inisialisasi simulasi
    # =====================================================
    oh = float(initial_inventory)

    pipeline = {}
    rows = []

    # =========================
    # 5. Fungsi QD
    # =========================
    def compute_qd_at(t_index):
        """
        QD:
        - actual_demand : demand hari ini + demand aktual H+1..H+DLT yang > OST
        - forecast      : demand hari ini + forecast H+1..H+DLT yang > OST
        """
        if t_index >= n:
            return 0.0

        qd_val = float(demands[t_index])

        if qd_source == "actual_demand":
            for kk in range(1, dlt + 1):
                j = t_index + kk
                if j < len(demands) and demands[j] > ost:
                    qd_val += float(demands[j])

        elif qd_source == "forecast":
            for kk in range(1, dlt + 1):
                j = t_index + kk
                if j < len(forecast) and forecast[j] > ost:
                    qd_val += float(forecast[j])

        else:
            raise ValueError("qd_source harus 'actual_demand' atau 'forecast'.")

        return qd_val
    # =========================
    # 6. Simulasi harian
    # =========================
    for t in range(n):
        date = pd.Timestamp(dates.iloc[t]).normalize()

        # 1. Receipt masuk
        receipt = float(pipeline.pop(date, 0.0))
        oh += receipt

        # 2. On-order / pipeline
        op = sum(
            float(q) for d, q in pipeline.items()
            if 1 <= (pd.Timestamp(d).normalize() - date).days <= dlt
        )

        # 3. Inventory position
        ip = oh + op

        # 4. QD hari ini
        qd = compute_qd_at(t)

        # 5. NFE tetap seperti Python sebelumnya: sebelum demand hari ini dikurangkan
        nfe = oh + op - qd

        # 6. Zona buffer
        if nfe <= tor:
            zone = "RED"
        elif nfe <= toy:
            zone = "YELLOW"
        else:
            zone = "GREEN"

        # 7. Preview OH setelah demand
        dem = float(demands[t])
        oh_before_demand = oh
        oh_after_preview = max(0.0, oh - min(dem, oh))

        if target_stock_basis == "ip":
            target_stock = ip

        elif target_stock_basis == "oh_before":
            target_stock = oh_before_demand

        elif target_stock_basis == "oh_after":
            target_stock = oh_after_preview

        else:
            raise ValueError("target_stock_basis harus 'ip', 'oh_before', atau 'oh_after'.")

        # 8. Keputusan order DDMRP Conditional
        q = 0
        q_raw = 0
        q_raw_target = 0
        qd_for_trigger = None
        order_reason = "NO_TRIGGER"

        candidate_trigger = (
            nfe <= toy
            and ip < target_level
        )

        if use_qd_next_for_trigger:
            qd_for_trigger = compute_qd_at(t)
        else:
            qd_for_trigger = qd

        qd_trigger = (
            candidate_trigger
                and qd_for_trigger > 0
        )

        if qd_trigger:
            q_raw_target = target_level - target_stock
            q_raw = max(0, q_raw_target)

            q = apply_moq_qmax(
                q_raw=q_raw,
                moq=moq,
                qmax=qmax,
                enforce_moq=True
            )

            if q > 0:
                if qmax is not None and qmax < moq:
                    order_reason = "CONDITIONAL_MOQ_OVERRIDE_QMAX"
                elif q_raw < moq:
                    order_reason = "CONDITIONAL_ADJUSTED_TO_MOQ"
                elif qmax is not None and qmax >= moq and q_raw > qmax:
                    order_reason = "CONDITIONAL_LIMITED_BY_QMAX"
                else:
                    order_reason = "CONDITIONAL_TARGET"
            else:
                order_reason = "NO_ORDER_TARGET_REACHED"

        # 9. Order masuk pipeline
        if q > 0:
            arr = (date + pd.Timedelta(days=dlt)).normalize()
            pipeline[arr] = pipeline.get(arr, 0.0) + float(q)

        # 7. Demand hari ini dipenuhi setelah keputusan order
        dem = float(demands[t])
        shipped = min(dem, oh)
        unmet = max(dem - shipped, 0.0)
        oh_end = oh - shipped

        # 8. Update OH untuk hari berikutnya
        oh = oh_end

        # 9. Total Biaya
        holding_cost = oh_end * hold_cost_per_unit_day
        order_cost_day = order_cost if q > 0 else 0
        penalty_cost = unmet * penalty_mult
        total_cost = holding_cost + order_cost_day + penalty_cost

        # -------------------------------------------------
        # 12. Simpan hasil harian
        # -------------------------------------------------
        rows.append({
            'date': date.date(),
            'method': "DDMRP_CONDITIONAL",

            'demand': round(dem, 2),
            'forecast': round(float(forecast[t]) if t < len(forecast) else 0, 2),
            'receipt': round(receipt, 2),

            'OH_before_demand': round(oh_end + shipped, 2),
            'OH_after_preview_for_target': round(oh_after_preview, 2),
            'OH_end': round(oh_end, 2),
            'OP': round(op, 2),
            'IP': round(ip, 2),

            'QD': round(qd, 2),
            'QD_for_trigger': round(qd_for_trigger, 2) if qd_for_trigger is not None else None,

            'NFE': round(nfe, 2),
            'TOR': round(tor, 2),
            'TOY': round(toy, 2),
            'TOG': round(tog, 2),

            'target_percentile': target_percentile,
            'target_level': round(target_level, 2),
            'avg_ltd': round(ltd_stats['avg_ltd'], 2),
            'target_ltd': round(ltd_stats['target_ltd'], 2),
            'target_stock_basis': target_stock_basis,
            'target_stock_used': round(target_stock, 2),
            'SS': ss_value,
            'qmax': qmax,

            'q_raw_target': round(q_raw_target, 2) if q_raw_target is not None else None,
            'q_raw': round(q_raw, 2) if q_raw is not None else None,

            'zone': zone,
            'order_reason': order_reason,
            'order_qty': int(q),

            'shipped': round(shipped, 2),
            'unmet': round(unmet, 2),

            'holding_cost': round(holding_cost, 2),
            'order_cost': round(order_cost_day, 2),
            'penalty_cost': round(penalty_cost, 2),
            'total_cost': round(total_cost, 2)
        })
    # =====================================================
    # 7. DataFrame hasil simulasi
    # =====================================================

    df = pd.DataFrame(rows)

    td = float(df["demand"].sum())
    ts = float(df["shipped"].sum())
    ns = int((df["unmet"] > 1e-6).sum())

    # =====================================================
    # 8. KPI
    # =====================================================

    kpi = {
        "method": "DDMRP_CONDITIONAL",

        "vf": round(vf, 4),
        "ltf": round(ltf, 4),
        "adu": round(adu, 4),

        "bzr": round(bzr, 2),
        "tor": round(tor, 2),
        "toy": round(toy, 2),
        "tog": round(tog, 2),

        "avg_ltd": round(ltd_stats["avg_ltd"], 4),
        "p75_ltd": round(ltd_stats["p75_ltd"], 4),
        "p90_ltd": round(ltd_stats["p90_ltd"], 4),
        "p95_ltd": round(ltd_stats["p95_ltd"], 4),
        "target_ltd": round(ltd_stats["target_ltd"], 4),
        "prob_ltd_pos": round(ltd_stats["prob_ltd_pos"], 4),

        "target_percentile": target_percentile,
        "target_level": round(target_level, 4),
        "ss": ss_value,

        "initial_inventory": round(float(initial_inventory), 4),

        "fill_rate": round(ts / td, 4) if td > 0 else 1.0,
        "csl": round(1 - ns / n, 4) if n > 0 else 1.0,
        "n_stockout": ns,

        "total_cost": round(float(df["total_cost"].sum()), 0),
        "hold_cost": round(float(df["holding_cost"].sum()), 0),
        "order_cost": round(float(df["order_cost"].sum()), 0),
        "penalty_cost": round(float(df["penalty_cost"].sum()), 0),

        "n_orders": int((df["order_qty"] > 0).sum()),
        "total_order_qty": int(df["order_qty"].sum()),
        "avg_oh": round(float(df["OH_end"].mean()), 2),

        "df_detail": df
    }

    # =====================================================
    # 9. Print ringkasan
    # =====================================================

    if verbose:
        print(f"\n{'=' * 60}")
        print("SIMULASI DDMRP_CONDITIONAL")
        print(f"{'=' * 60}")
        print(f"Initial Inventory = {initial_inventory}")
        print(f"ADU               = {adu:.4f}")
        print(f"VF                = {vf:.4f}")
        print(f"LTF               = {ltf:.4f}")
        print(f"TOR               = {tor:.2f}")
        print(f"TOY               = {toy:.2f}")
        print(f"TOG               = {tog:.2f}")
        print(f"Target Percentile = {target_percentile}")
        print(f"Target Level      = {target_level:.2f}")
        print(f"Safety Stock      = {ss_value}")
        print(f"Fill Rate         = {kpi['fill_rate'] * 100:.2f}%")
        print(f"CSL               = {kpi['csl'] * 100:.2f}%")
        print(f"Stockout Days     = {kpi['n_stockout']}")
        print(f"Jumlah Order      = {kpi['n_orders']}")
        print(f"Total Qty Order   = {kpi['total_order_qty']}")
        print(f"Total Cost        = Rp{kpi['total_cost']:,.0f}")

    return kpi

## [PART 5] — Optimasi Buffer
---



In [185]:
import numpy as np
import pandas as pd
import math


def is_missing(x):
    """
    Mengecek nilai kosong/NaN.
    """
    if x is None:
        return True
    try:
        return bool(pd.isna(x))
    except Exception:
        return False


def normalize_qmax(qmax):
    """
    Normalisasi Qmax.
    Jika kosong/NaN/<=0, dianggap tidak ada Qmax.
    """
    if is_missing(qmax):
        return None

    qmax = float(qmax)

    if qmax <= 0:
        return None

    return qmax


def normalize_moq(moq, default=0):
    """
    Normalisasi MOQ.
    Jika kosong, gunakan default.
    """
    if is_missing(moq):
        return int(default)

    return max(int(float(moq)), 0)


VF_GLOBAL_BOUNDS = (0.01, 3.00)
LTF_GLOBAL_BOUNDS = (0.01, 3.00)

# ============================================================
# 3. MEMBENTUK BUFFER DARI DATA REAL
# ============================================================

def build_buffer(demands, vf, ltf, params):
    """
    Menghitung buffer DDMRP dari data demand, VF, dan LTF.
    """

    demands = np.asarray(demands, dtype=float)

    dlt = max(int(params["dlt"]), 1)
    moq = normalize_moq(params.get("moq", 0), default=0)

    adu = float(np.mean(demands)) if len(demands) > 0 else 0.0
    ost = adu

    bzr = adu * dlt * ltf
    tor = bzr * vf
    yellow = adu * dlt
    green = max(bzr, moq)

    toy = tor + yellow
    tog = toy + green

    buffer = {
        "vf": float(vf),
        "ltf": float(ltf),

        "adu": float(adu),
        "ost": float(ost),

        "bzr": float(bzr),
        "tor": float(tor),
        "yellow": float(yellow),
        "green": float(green),
        "toy": float(toy),
        "tog": float(tog),
    }

    return buffer


# ============================================================
# 4. FITNESS FUNCTION UNTUK GA - PURE DDMRP
# ============================================================

def fitness_buffer_ga(vf, ltf, demands, dates, params):
    """
    Fitness function untuk optimasi buffer DDMRP.

    GA mencari kombinasi:
    - VF
    - LTF

    Setiap kandidat VF dan LTF:
    1. Dibentuk buffer DDMRP
    2. Disimulasikan dengan simulate_ddmrp()
    3. Dihitung total cost
    4. Jika service level tidak tercapai, diberi penalti
    """

    # 1. Bentuk buffer DDMRP
    buffer = build_buffer(
        demands=demands,
        vf=vf,
        ltf=ltf,
        params=params
    )

    # 2. Simulasi DDMRP standar
    kpi = simulate_ddmrp(
        demands=demands,
        dates=dates,

        vf=vf,
        ltf=ltf,
        dlt=params["dlt"],
        pack_size=params.get("pack_size", 1),

        unit_price=params.get("price_ea", 0),
        hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
        order_cost=params["order_cost"],
        penalty_mult=params["penalty_per_unit"],

        qmax=params.get("qmax", None),
        initial_inventory=params["initial_inventory"],
        moq=params.get("moq", 0),

        verbose=False
    )

    # 3. Ambil KPI
    total_cost = kpi["total_cost"]
    fill_rate = kpi["fill_rate"]
    csl = kpi["csl"]

    target_sl = params.get("target_sl", 0.95)

    # 4. Penalti service level
    service_penalty = 0

    if fill_rate < target_sl:
        service_penalty += (target_sl - fill_rate) * 1_000_000_000

    if csl < target_sl:
        service_penalty += (target_sl - csl) * 500_000_000

    fitness = total_cost + service_penalty

    return fitness, kpi, buffer



# ============================================================
# 5. GENETIC ALGORITHM UNTUK OPTIMASI BUFFER DDMRP
# ============================================================

def run_ga_buffer_optimization(
    demands,
    dates,
    params,
    cls,
    pop_size=80,
    n_gen=150,
    mutation_rate=0.45,
    elite_size=5,
    random_state=42,
    verbose=True
):
    """
    Genetic Algorithm untuk mencari VF dan LTF terbaik.
    """

    np.random.seed(random_state)

    # ========================================================
    # 1. Batas optimasi global
    # ========================================================
    vf_low, vf_high = VF_GLOBAL_BOUNDS
    ltf_low, ltf_high = LTF_GLOBAL_BOUNDS

    if vf_low >= vf_high:
        raise ValueError("vf_low harus lebih kecil dari vf_high.")

    if ltf_low >= ltf_high:
        raise ValueError("ltf_low harus lebih kecil dari ltf_high.")

    pop_size = max(int(pop_size), 4)
    n_gen = max(int(n_gen), 1)
    elite_size = max(1, min(int(elite_size), pop_size - 1))

    # ========================================================
    # 2. Ambil initial VF dan LTF dari hasil klasifikasi
    # ========================================================
    vf_init = float(cls.get("vf_init", 0.50))
    ltf_init = float(cls.get("ltf_init", 0.50))

    vf_init = float(np.clip(vf_init, vf_low, vf_high))
    ltf_init = float(np.clip(ltf_init, ltf_low, ltf_high))

    # ========================================================
    # 3. Inisialisasi populasi
    # ========================================================
    population = []

    # Kandidat pertama: nilai awal hasil klasifikasi
    population.append([vf_init, ltf_init])

    # Kandidat di sekitar nilai awal
    n_near_init = int(0.30 * pop_size)

    for _ in range(n_near_init):
        cand_vf = vf_init + np.random.normal(0, 0.15)
        cand_ltf = ltf_init + np.random.normal(0, 0.15)

        cand_vf = float(np.clip(cand_vf, vf_low, vf_high))
        cand_ltf = float(np.clip(cand_ltf, ltf_low, ltf_high))

        population.append([cand_vf, cand_ltf])

    # Sisanya random global
    while len(population) < pop_size:
        cand_vf = np.random.uniform(vf_low, vf_high)
        cand_ltf = np.random.uniform(ltf_low, ltf_high)
        population.append([cand_vf, cand_ltf])

    population = np.array(population)

    best_solution = None
    best_fitness = np.inf
    best_kpi = None
    best_buffer = None

    history = []

    # ========================================================
    # 4. Loop generasi GA
    # ========================================================
    for gen in range(n_gen):

        evaluated = []

        for vf, ltf in population:

            fitness, kpi, buffer = fitness_buffer_ga(
                vf=vf,
                ltf=ltf,
                demands=demands,
                dates=dates,
                params=params
            )

            evaluated.append({
                "vf": vf,
                "ltf": ltf,
                "fitness": fitness,
                "kpi": kpi,
                "buffer": buffer
            })

            if fitness < best_fitness:
                best_fitness = fitness
                best_solution = [vf, ltf]
                best_kpi = kpi
                best_buffer = buffer

        evaluated = sorted(evaluated, key=lambda x: x["fitness"])
        best_gen = evaluated[0]

        history.append({
            "generation": gen + 1,

            "best_fitness_generation": best_gen["fitness"],
            "best_vf_generation": best_gen["vf"],
            "best_ltf_generation": best_gen["ltf"],
            "best_total_cost_generation": best_gen["kpi"]["total_cost"],
            "best_fill_rate_generation": best_gen["kpi"]["fill_rate"],
            "best_csl_generation": best_gen["kpi"]["csl"],

            "best_fitness_global": best_fitness,
            "best_vf_global": best_solution[0],
            "best_ltf_global": best_solution[1],
            "best_total_cost_global": best_kpi["total_cost"],
            "best_fill_rate_global": best_kpi["fill_rate"],
            "best_csl_global": best_kpi["csl"],

            "vf_init": vf_init,
            "ltf_init": ltf_init,
            "vf_low": vf_low,
            "vf_high": vf_high,
            "ltf_low": ltf_low,
            "ltf_high": ltf_high
        })

        # ====================================================
        # 5. Elitism
        # ====================================================
        elites = evaluated[:elite_size]
        new_population = [[e["vf"], e["ltf"]] for e in elites]

        parent_pool_size = max(2, pop_size // 2)
        parent_pool = evaluated[:parent_pool_size]

        # ====================================================
        # 6. Crossover dan mutation
        # ====================================================
        vf_range = vf_high - vf_low
        ltf_range = ltf_high - ltf_low

        while len(new_population) < pop_size:

            p1 = parent_pool[np.random.randint(0, len(parent_pool))]
            p2 = parent_pool[np.random.randint(0, len(parent_pool))]

            alpha = np.random.rand()

            child_vf = alpha * p1["vf"] + (1 - alpha) * p2["vf"]
            child_ltf = alpha * p1["ltf"] + (1 - alpha) * p2["ltf"]

            if np.random.rand() < mutation_rate:
                child_vf += np.random.normal(0, 0.20 * vf_range)

            if np.random.rand() < mutation_rate:
                child_ltf += np.random.normal(0, 0.20 * ltf_range)

            child_vf = float(np.clip(child_vf, vf_low, vf_high))
            child_ltf = float(np.clip(child_ltf, ltf_low, ltf_high))

            new_population.append([child_vf, child_ltf])

        population = np.array(new_population)

        if verbose:
            print(
                f"Gen {gen + 1:03d} | "
                f"Fitness={best_fitness:,.0f} | "
                f"VF={best_solution[0]:.4f} | "
                f"LTF={best_solution[1]:.4f} | "
                f"Cost={best_kpi['total_cost']:,.0f} | "
                f"FR={best_kpi['fill_rate'] * 100:.2f}% | "
                f"CSL={best_kpi['csl'] * 100:.2f}%"
            )

    result = {
        "vf_opt": best_solution[0],
        "ltf_opt": best_solution[1],
        "fitness": best_fitness,
        "kpi": best_kpi,
        "buffer": best_buffer,
        "history": pd.DataFrame(history)
    }

    return result

## [RUN 1] — Load & Eksplorasi Data

In [186]:
# Load data
data = load_all_data(FILE_PATH)
all_skus = sorted(data['sales']['ID Item'].unique().tolist())
print(f"\nTotal SKU: {len(all_skus)}")

# Daftar SKU
sku_list = get_sku_list(data)

Loading: /content/sample_data/Data 2.xlsx
✅ Sales    : 13,679 baris | 45 SKU | 2025-06-01 s/d 2026-03-31
✅ Master   : 45 SKU

Total SKU: 45

=== DAFTAR SKU ===
  ID Item                                      Grup  Tgl_Mulai  Tgl_Akhir  Jml_Hari  Total_Demand  ADU  Lead Time_Days  Logistic Cost/Order
100004821            HYDRAULIC PUMP ASSY - CAT 740B 2025-06-02 2026-03-31       303             4  0.0              63               100000
100004822         FINAL DRIVE MOTOR - KOMATSU PC400 2025-06-01 2026-03-31       304             2  0.0              78               100000
100004823        SWING BEARING ASSY - HITACHI EX500 2025-06-01 2026-03-31       304             4  0.0              93               100000
100004824       ENGINE OVERHAUL KIT - CUMMINS QSK19 2025-06-01 2026-03-31       304             4  0.0              65               100000
100004825      TORQUE CONVERTER ASSY - ALLISON 4700 2025-06-01 2026-03-31       304             5  0.0              78               100000


In [187]:
SKU = sku_number   # atau isi manual, misalnya SKU = 100123

def parse_qmax(x):
    """
    Membaca Qmax dari Excel.
    Kosong / None / '-' dianggap tidak ada batas Qmax.
    """
    if x is None or pd.isna(x):
        return None

    if isinstance(x, str):
        x_clean = x.strip().lower()
        if x_clean in ["", "none", "nan", "-", "tidak", "no"]:
            return None

    try:
        qmax = float(x)
        if qmax <= 0:
            return None
        return qmax
    except:
        return None


def parse_target_percentile(x, default=0.95):
    """
    Membaca target percentile dari Excel.
    95   -> 0.95
    0.95 -> 0.95
    """
    if x is None or pd.isna(x):
        return default

    if isinstance(x, str):
        x = x.strip().replace("%", "")
        if x == "":
            return default

    try:
        val = float(x)
        if val > 1:
            val = val / 100.0
        return min(max(val, 0.01), 0.999)
    except:
        return default


def parse_moq(x, default=0):
    """
    Membaca MOQ dari Excel.
    """
    if x is None or pd.isna(x):
        return default

    if isinstance(x, str):
        x_clean = x.strip().lower()
        if x_clean in ["", "none", "nan", "-"]:
            return default
        x = x_clean

    try:
        return max(int(float(x)), 0)
    except:
        return default

params = get_sku_params(data, SKU)

params["qmax"] = parse_qmax(params.get("qmax", None))
params["moq"] = parse_moq(params.get("moq", 0), default=0)
params["target_sl"] = params.get("target_sl", 0.95)

if params.get("initial_inventory", None) is None or pd.isna(params.get("initial_inventory", None)):
    raise ValueError("Initial inventory kosong. Cek kembali input master Excel.")

params["dlt"] = max(int(params.get("dlt", 1)), 1)

print("Initial Inventory :", params["initial_inventory"])
print("Qmax              :", params["qmax"])
print("MOQ               :", params["moq"])
print("DLT               :", params["dlt"])
print("Target SL         :", params["target_sl"])

Initial Inventory : 2.0
Qmax              : 1.0
MOQ               : 1
DLT               : 47
Target SL         : 0.95


In [188]:
df_clf = get_sku_demand(data, SKU)

s = df_clf["Demand"].values.astype(float)

if (s > 0).sum() > 0:
    adu = float(np.mean(s[s > 0]))
else:
    adu = 0.0

clean = s

clf = classify_sku(df_clf, params, verbose=True)

print("Method:", clf["method"])
print("qmax  :", params["qmax"])
print("moq   :", params["moq"])

clean_train = df_clf["Demand"].values.astype(float)
dates_train = df_clf["Date"].reset_index(drop=True)

print("Jumlah data train:", len(clean_train))
print("Tanggal awal     :", dates_train.iloc[0])
print("Tanggal akhir    :", dates_train.iloc[-1])


# ============================================================
# BASELINE SEBELUM GA
# ============================================================

vf_base = clf["vf_init"]
ltf_base = clf["ltf_init"]

buffer_base = build_buffer(
    demands=clean_train,
    vf=vf_base,
    ltf=ltf_base,
    params=params
)

kpi_base = simulate_ddmrp(
    demands=clean_train,
    dates=dates_train,

    vf=vf_base,
    ltf=ltf_base,
    dlt=params["dlt"],
    pack_size=params.get("pack_size", 1),

    unit_price=params.get("price_ea", 0),
    hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
    order_cost=params["order_cost"],
    penalty_mult=params["penalty_per_unit"],

    qmax=params.get("qmax", None),
    initial_inventory=params["initial_inventory"],
    moq=params.get("moq", 0),

    verbose=True
)

print("\n" + "=" * 70)
print("BASELINE BUFFER SEBELUM GA")
print("=" * 70)
print(f"SKU              : {SKU}")
print(f"VF baseline      : {vf_base:.4f}")
print(f"LTF baseline     : {ltf_base:.4f}")
print(f"ADU              : {buffer_base['adu']:.4f}")
print(f"BZR              : {buffer_base['bzr']:.2f}")
print(f"TOR              : {buffer_base['tor']:.2f}")
print(f"YELLOW           : {buffer_base['yellow']:.2f}")
print(f"GREEN            : {buffer_base['green']:.2f}")
print(f"TOY              : {buffer_base['toy']:.2f}")
print(f"TOG              : {buffer_base['tog']:.2f}")
print(f"Fill Rate        : {kpi_base['fill_rate'] * 100:.2f}%")
print(f"CSL              : {kpi_base['csl'] * 100:.2f}%")
print(f"Total Cost       : Rp{kpi_base['total_cost']:,.0f}")



# ============================================================
# JALANKAN OPTIMASI BUFFER DDMRP-GA
# ============================================================

ga_result = run_ga_buffer_optimization(
    demands=clean_train,
    dates=dates_train,
    params=params,
    cls=clf,
    pop_size=30,
    n_gen=50,
    mutation_rate=0.20,
    elite_size=3,
    random_state=42,
    verbose=True
)

vf_opt = ga_result["vf_opt"]
ltf_opt = ga_result["ltf_opt"]
buffer_opt = ga_result["buffer"]
kpi_opt = ga_result["kpi"]
history_ga = ga_result["history"]



print("\n" + "=" * 70)
print("HASIL OPTIMASI BUFFER DDMRP-GA")
print("=" * 70)

print(f"SKU              : {SKU}")
print(f"VF optimal       : {vf_opt:.4f}")
print(f"LTF optimal      : {ltf_opt:.4f}")
print(f"ADU              : {buffer_opt['adu']:.4f}")
print(f"BZR optimal      : {buffer_opt['bzr']:.2f}")
print(f"TOR optimal      : {buffer_opt['tor']:.2f}")
print(f"YELLOW optimal   : {buffer_opt['yellow']:.2f}")
print(f"GREEN optimal    : {buffer_opt['green']:.2f}")
print(f"TOY optimal      : {buffer_opt['toy']:.2f}")
print(f"TOG optimal      : {buffer_opt['tog']:.2f}")

print("\n--- KPI Hasil GA ---")
print(f"Fill Rate        : {kpi_opt['fill_rate'] * 100:.2f}%")
print(f"CSL              : {kpi_opt['csl'] * 100:.2f}%")
print(f"Stockout Days    : {kpi_opt['n_stockout']}")
print(f"Jumlah Order     : {kpi_opt['n_orders']}")
print(f"Total Qty Order  : {kpi_opt['total_order_qty']}")
print(f"Holding Cost     : Rp{kpi_opt['hold_cost']:,.0f}")
print(f"Order Cost       : Rp{kpi_opt['order_cost']:,.0f}")
print(f"Total Cost       : Rp{kpi_opt['total_cost']:,.0f}")


✅ SKU 100008503 | 2025-06-01 s/d 2026-03-31 | 304 hari | demand total: 4.000 Pcs

  KLASIFIKASI SKU 100008503 | Spare Part
  ADI   = 76.0000
  CV²   = 0.0000
  Kategori: 🟠 INTERMITTENT
  Metode  : DDMRP_CONDITIONAL
  DLT     : 47 hari (LONG)
  qmax    : 1.0
  Target Percentile: 0.98
  Search space: VF=[0.30,0.65] | LTF=[0.20,0.40]
Method: DDMRP_CONDITIONAL
qmax  : 1.0
moq   : 1
Jumlah data train: 304
Tanggal awal     : 2025-06-01 00:00:00
Tanggal akhir    : 2026-03-31 00:00:00

SIMULASI DDMRP
Initial Inventory = 2.0
ADU               = 0.0132
VF                = 0.4750
LTF               = 0.3000
TOR               = 0.09
TOY               = 0.71
TOG               = 1.71
Fill Rate         = 100.00%
CSL               = 100.00%
Stockout Days     = 0
Jumlah Order      = 2
Total Qty Order   = 3
Total Cost        = Rp4,544,997

BASELINE BUFFER SEBELUM GA
SKU              : 100008503
VF baseline      : 0.4750
LTF baseline     : 0.3000
ADU              : 0.0132
BZR              : 0.19
TOR      

In [189]:
def compute_ltd_stats(demands, dlt, target_percentile=0.95):
    """
    Menghitung rolling Lead Time Demand untuk DDMRP Conditional.
    """

    s = np.asarray(demands, dtype=float)
    n = len(s)
    dlt = int(dlt)

    if n == 0:
        ltd = np.array([0.0])

    elif n < dlt:
        ltd = np.array([s.sum()])

    else:
        ltd = np.array([
            s[i:i + dlt].sum()
            for i in range(n - dlt + 1)
        ])

    avg_ltd = float(np.mean(ltd)) if len(ltd) > 0 else 0.0

    p75_ltd = float(np.percentile(ltd, 75)) if len(ltd) > 0 else 0.0
    p90_ltd = float(np.percentile(ltd, 90)) if len(ltd) > 0 else 0.0
    p95_ltd = float(np.percentile(ltd, 95)) if len(ltd) > 0 else 0.0

    target_ltd = float(
        np.percentile(ltd, target_percentile * 100)
    ) if len(ltd) > 0 else 0.0

    prob_ltd_pos = float(np.mean(ltd > 0)) if len(ltd) > 0 else 0.0

    ss_target = math.ceil(max(0, target_ltd - avg_ltd))
    ss_p90 = math.ceil(max(0, p90_ltd - avg_ltd))
    ss_p95 = math.ceil(max(0, p95_ltd - avg_ltd))

    return {
        "ltd_values": ltd,
        "avg_ltd": avg_ltd,
        "p75_ltd": p75_ltd,
        "p90_ltd": p90_ltd,
        "p95_ltd": p95_ltd,
        "target_ltd": target_ltd,
        "prob_ltd_pos": prob_ltd_pos,
        "ss_target": ss_target,
        "ss_p90": ss_p90,
        "ss_p95": ss_p95
    }

def build_target_level(demands, params):
    """
    Menghitung target level untuk DDMRP Conditional.
    """

    demands = np.asarray(demands, dtype=float)

    dlt = max(int(params["dlt"]), 1)

    target_percentile = params.get("target_percentile", 0.95)

    if target_percentile > 1:
        target_percentile = target_percentile / 100.0

    target_percentile = min(max(target_percentile, 0.01), 0.999)

    ltd_stats = compute_ltd_stats(
        demands=demands,
        dlt=dlt,
        target_percentile=target_percentile
    )

    target_level = math.ceil(ltd_stats["target_ltd"])

    return {
        "target_percentile": target_percentile,
        "target_level": float(target_level),
        "avg_ltd": float(ltd_stats["avg_ltd"]),
        "p75_ltd": float(ltd_stats["p75_ltd"]),
        "p90_ltd": float(ltd_stats["p90_ltd"]),
        "p95_ltd": float(ltd_stats["p95_ltd"]),
        "target_ltd": float(ltd_stats["target_ltd"]),
        "ss": float(ltd_stats["ss_target"])
    }

In [190]:
def simulate_selected_method(
    demands,
    dates,
    params,
    clf,
    vf=None,
    ltf=None,
    verbose=True
):
    """
    Menjalankan simulasi harian berdasarkan hasil klasifikasi SKU.

    Jika clf["method"] = "DDMRP":
        menjalankan simulate_ddmrp()

    Jika clf["method"] = "DDMRP_CONDITIONAL":
        menjalankan simulate_ddmrp_conditional()
    """

    method = clf["method"]

    # Jika VF dan LTF tidak diberikan, gunakan hasil klasifikasi
    if vf is None:
        vf = clf["vf_init"]

    if ltf is None:
        ltf = clf["ltf_init"]

    print("\n" + "=" * 70)
    print("SIMULASI HARIAN BERDASARKAN HASIL KLASIFIKASI")
    print("=" * 70)
    print(f"Method : {method}")
    print(f"VF     : {vf:.4f}")
    print(f"LTF    : {ltf:.4f}")

    # ========================================================
    # Jika metode DDMRP standar
    # ========================================================
    if method == "DDMRP":

        kpi = simulate_ddmrp(
            demands=demands,
            dates=dates,

            vf=vf,
            ltf=ltf,
            dlt=params["dlt"],
            pack_size=params.get("pack_size", 1),

            unit_price=params.get("price_ea", 0),
            hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
            order_cost=params["order_cost"],
            penalty_mult=params["penalty_per_unit"],

            qmax=params.get("qmax", None),
            initial_inventory=params["initial_inventory"],
            moq=params.get("moq", 0),

            verbose=verbose
        )

    # ========================================================
    # Jika metode DDMRP Conditional
    # ========================================================
    elif method == "DDMRP_CONDITIONAL":

        target_info = build_target_level(
            demands=demands,
            params=params
        )

        kpi = simulate_ddmrp_conditional(
            demands=demands,
            dates=dates,

            vf=vf,
            ltf=ltf,
            dlt=params["dlt"],
            pack_size=params.get("pack_size", 1),

            unit_price=params.get("price_ea", 0),
            hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
            order_cost=params["order_cost"],
            penalty_mult=params["penalty_per_unit"],

            qmax=params.get("qmax", None),
            initial_inventory=params["initial_inventory"],
            moq=params.get("moq", 0),

            target_level=target_info["target_level"],
            target_percentile=target_info["target_percentile"],

            qd_source="actual_demand",
            target_stock_basis="ip",
            use_qd_next_for_trigger=True,

            verbose=verbose
        )

        kpi["target_level"] = target_info["target_level"]
        kpi["target_percentile"] = target_info["target_percentile"]

    else:
        raise ValueError("Method tidak dikenali. Gunakan 'DDMRP' atau 'DDMRP_CONDITIONAL'.")

    return kpi

In [191]:
def simulate_selected_method(
    demands,
    dates,
    params,
    clf,
    vf=None,
    ltf=None,
    verbose=True
):
    """
    Menjalankan simulasi harian berdasarkan hasil klasifikasi SKU.

    Jika clf["method"] = "DDMRP":
        menjalankan simulate_ddmrp()

    Jika clf["method"] = "DDMRP_CONDITIONAL":
        menjalankan simulate_ddmrp_conditional()
    """

    method = clf["method"]

    # Jika VF dan LTF tidak diberikan, gunakan hasil klasifikasi
    if vf is None:
        vf = clf["vf_init"]

    if ltf is None:
        ltf = clf["ltf_init"]

    print("\n" + "=" * 70)
    print("SIMULASI HARIAN BERDASARKAN HASIL KLASIFIKASI")
    print("=" * 70)
    print(f"Method : {method}")
    print(f"VF     : {vf:.4f}")
    print(f"LTF    : {ltf:.4f}")

    # ========================================================
    # Jika metode DDMRP standar
    # ========================================================
    if method == "DDMRP":

        kpi = simulate_ddmrp(
            demands=demands,
            dates=dates,

            vf=vf,
            ltf=ltf,
            dlt=params["dlt"],
            pack_size=params.get("pack_size", 1),

            unit_price=params.get("price_ea", 0),
            hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
            order_cost=params["order_cost"],
            penalty_mult=params["penalty_per_unit"],

            qmax=params.get("qmax", None),
            initial_inventory=params["initial_inventory"],
            moq=params.get("moq", 0),

            verbose=verbose
        )

    # ========================================================
    # Jika metode DDMRP Conditional
    # ========================================================
    elif method == "DDMRP_CONDITIONAL":

        target_info = build_target_level(
            demands=demands,
            params=params
        )

        kpi = simulate_ddmrp_conditional(
            demands=demands,
            dates=dates,

            vf=vf,
            ltf=ltf,
            dlt=params["dlt"],
            pack_size=params.get("pack_size", 1),

            unit_price=params.get("price_ea", 0),
            hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
            order_cost=params["order_cost"],
            penalty_mult=params["penalty_per_unit"],

            qmax=params.get("qmax", None),
            initial_inventory=params["initial_inventory"],
            moq=params.get("moq", 0),

            target_level=target_info["target_level"],
            target_percentile=target_info["target_percentile"],

            qd_source="actual_demand",
            target_stock_basis="ip",
            use_qd_next_for_trigger=True,

            verbose=verbose
        )

        kpi["target_level"] = target_info["target_level"]
        kpi["target_percentile"] = target_info["target_percentile"]

    else:
        raise ValueError("Method tidak dikenali. Gunakan 'DDMRP' atau 'DDMRP_CONDITIONAL'.")

    return kpi

In [192]:
# ============================================================
# SIMULASI HARIAN DDMRP HASIL OPTIMASI BUFFER
# ============================================================

vf_opt = ga_result["vf_opt"]
ltf_opt = ga_result["ltf_opt"]

kpi_daily = simulate_selected_method(
    demands=clean_train,
    dates=dates_train,
    params=params,
    clf=clf,
    verbose=True
)

df_daily = kpi_daily["df_detail"]

display(df_daily)

kpi_opt_daily = simulate_ddmrp(
    demands=clean_train,
    dates=dates_train,

    vf=vf_opt,
    ltf=ltf_opt,
    dlt=params["dlt"],
    pack_size=params.get("pack_size", 1),

    unit_price=params.get("price_ea", 0),
    hold_cost_per_unit_day=params["hold_cost_per_unit_day"],
    order_cost=params["order_cost"],
    penalty_mult=params["penalty_per_unit"],

    qmax=params.get("qmax", None),
    initial_inventory=params["initial_inventory"],
    moq=params.get("moq", 0),

    verbose=True
)

df_daily_opt = kpi_opt_daily["df_detail"]

display(df_daily_opt.head())


SIMULASI HARIAN BERDASARKAN HASIL KLASIFIKASI
Method : DDMRP_CONDITIONAL
VF     : 0.4750
LTF    : 0.3000

SIMULASI DDMRP_CONDITIONAL
Initial Inventory = 2.0
ADU               = 0.0132
VF                = 0.4750
LTF               = 0.3000
TOR               = 0.09
TOY               = 0.71
TOG               = 1.71
Target Percentile = 0.98
Target Level      = 2.00
Safety Stock      = None
Fill Rate         = 100.00%
CSL               = 100.00%
Stockout Days     = 0
Jumlah Order      = 3
Total Qty Order   = 3
Total Cost        = Rp3,682,598


,date,method,demand,forecast,receipt,OH_before_demand,OH_after_preview_for_target,OH_end,OP,IP,QD,QD_for_trigger,NFE,TOR,TOY,TOG,target_percentile,target_level,avg_ltd,target_ltd,target_stock_basis,target_stock_used,SS,qmax,q_raw_target,q_raw,zone,order_reason,order_qty,shipped,unmet,holding_cost,order_cost,penalty_cost,total_cost
0,2025-06-01,DDMRP_CONDITIONAL,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,2.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
1,2025-06-02,DDMRP_CONDITIONAL,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,2.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
2,2025-06-03,DDMRP_CONDITIONAL,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,2.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
3,2025-06-04,DDMRP_CONDITIONAL,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,2.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
4,2025-06-05,DDMRP_CONDITIONAL,0.0,0.0,0.0,2.0,2.0,2.0,0.0,2.0,0.0,0.0,2.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,2.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299,2026-03-27,DDMRP_CONDITIONAL,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,1.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,9068.63,0.0,0.0,9068.63
300,2026-03-28,DDMRP_CONDITIONAL,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,1.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,9068.63,0.0,0.0,9068.63
301,2026-03-29,DDMRP_CONDITIONAL,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,1.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,9068.63,0.0,0.0,9068.63
302,2026-03-30,DDMRP_CONDITIONAL,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.09,0.71,1.71,0.98,2.0,0.62,2.0,ip,1.0,None,1.0,0.0,0.0,GREEN,NO_TRIGGER,0,0.0,0.0,9068.63,0.0,0.0,9068.63



SIMULASI DDMRP
Initial Inventory = 2.0
ADU               = 0.0132
VF                = 0.0100
LTF               = 0.0100
TOR               = 0.00
TOY               = 0.62
TOG               = 1.62
Fill Rate         = 100.00%
CSL               = 100.00%
Stockout Days     = 0
Jumlah Order      = 2
Total Qty Order   = 3
Total Cost        = Rp4,395,657


,date,method,demand,forecast,receipt,OH_before_demand,OH_end,OP,IP,QD,NFE,TOR,TOY,TOG,zone,q_raw,order_reason,order_qty,shipped,unmet,holding_cost,order_cost,penalty_cost,total_cost
0,2025-06-01,DDMRP,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,2.0,0.0,0.62,1.62,GREEN,0.0,NO_ORDER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
1,2025-06-02,DDMRP,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,2.0,0.0,0.62,1.62,GREEN,0.0,NO_ORDER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
2,2025-06-03,DDMRP,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,2.0,0.0,0.62,1.62,GREEN,0.0,NO_ORDER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
3,2025-06-04,DDMRP,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,2.0,0.0,0.62,1.62,GREEN,0.0,NO_ORDER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
4,2025-06-05,DDMRP,0.0,0.0,0.0,2.0,2.0,0.0,2.0,0.0,2.0,0.0,0.62,1.62,GREEN,0.0,NO_ORDER,0,0.0,0.0,18137.25,0.0,0.0,18137.25
